# VD 12 — Module 11 on two years of 5-minute history, with unrestricted-memory and climatology benchmarks

**Branch:** `Volatility_Dynamics` · **Discussion:** `discussions/Volatility_Dynamics/026_claude_module12_long_history_preregistration.md` · **Code:** `volatility_lifted_memory_long.py` (new) on top of `volatility_lifted_memory.py` v0.1.1 (unchanged)

Module 11 returned **INSUFFICIENT_DATA** on 21 days of research-DB returns (5 test days, 15 unique onset times). Module 12 asks the same question with the same pipeline on about two years of Binance-spot 5m closes (`DERIBIT_DATA_ROOT/history/price_5m`). Two benchmarks are added:

| Model | Features beyond asset and clock controls |
|---|---|
| **K — climatology** (new) | none (asset fixed effects + time-of-day harmonics only) |
| A — current | current de-clocked log variance |
| B — fixed lags | current + 5, 10, 30, 60, 120 min lags |
| E — two EMAs | current + 30 min and 4 h EMAs |
| F — equal multiscale | current + equal-weight mean of the factor grid + fast–slow |
| C — rough-inspired lift | current + kernel-weighted memory + fast–slow |
| **U — unrestricted** (new) | current + every factor EMA as its own regressor ($N$ chosen on validation) |

U nests every fixed kernel weighting of the same factors, so it is the ceiling for "shape of the memory kernel".

**Sample (pre-registered):** data strictly before 2026-09-19 00:00 UTC; the Module 11 window is excluded. Assets with fewer than 365 days are excluded. Split 60/20/20 by UTC day. All other rules as Module 11.

**Decision (60 min, paired UTC-day bootstrap, 1,000 reps):**
`PASS_ROUGH_SPECIFIC` if C beats B, E and F; else `MULTISCALE_MEMORY_NOT_ROUGH_SPECIFIC` if U beats E; else `TWO_EMA_MEMORY_SUFFICIENT` if E beats B; else `SKILL_WITHOUT_LONG_MEMORY` if any model beats K; else `NO_SKILL`. "Beats" means the mean log-loss difference is < 0 **and** its upper 97.5% bootstrap bound is < 0. The Module 11 rule (C beats B and F) is reported alongside for continuity.

Environment (optional): `VD12_HISTORY_DIR`, `VD12_ASSETS`, `VD12_BOOT`.

In [1]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT/'src'/'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT/'src'/'data_engineering').exists(), 'Run inside deribit-market-data repository'
load_dotenv(PROJECT_ROOT/'.env'); load_dotenv()
SRC = PROJECT_ROOT/'src'
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
from data_engineering.modeling import volatility_lifted_memory as lm
from data_engineering.modeling import volatility_lifted_memory_long as ll

DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
HIST_DIR = Path(os.getenv('VD12_HISTORY_DIR', str(DATA_ROOT/'history'/'price_5m')))
ASSETS = [x.strip().upper() for x in os.getenv('VD12_ASSETS', 'AVAX,BTC,ETH,HYPE,SOL,TRX,XRP').split(',') if x.strip()]
BOOT = int(os.getenv('VD12_BOOT', str(ll.BOOT)))
OUT = PROJECT_ROOT/'notebooks'/'Volatility_Dynamics'/'12'
OUT.mkdir(parents=True, exist_ok=True)
print('VD 12', ll.VERSION, ll.selftest(), '| history:', HIST_DIR, '| bootstrap reps:', BOOT)
print(json.dumps(ll.registration(), indent=2))
T0 = time.time()

VD 12 0.1.0 {'version': '0.1.0', 'ok': True} | history: D:\MarketData\Deribit\history\price_5m | bootstrap reps: 1000
{
  "version": "0.1.0",
  "base_module": "volatility_lifted_memory",
  "sample_end_exclusive": "2026-09-19 00:00:00+00:00",
  "min_days": 365,
  "boot": 1000,
  "models": [
    "K_climatology",
    "A_current",
    "B_fixed",
    "E_exponential",
    "F_equal_multiscale",
    "C_lifted",
    "U_unrestricted"
  ],
  "h_grid": [
    0.05,
    0.1,
    0.2,
    0.3,
    0.4
  ],
  "n_grid": [
    4,
    6,
    8
  ],
  "ridge_grid": [
    0.0001,
    0.001,
    0.01
  ],
  "horizons": [
    30,
    60,
    90,
    120
  ],
  "primary_horizon": 60,
  "history_coverage": 0.8,
  "status_order": [
    "INSUFFICIENT_DATA",
    "PASS_ROUGH_SPECIFIC",
    "MULTISCALE_MEMORY_NOT_ROUGH_SPECIFIC",
    "TWO_EMA_MEMORY_SUFFICIENT",
    "SKILL_WITHOUT_LONG_MEMORY",
    "NO_SKILL"
  ]
}


## 1. Two-year 5-minute panel

In [2]:
raw, excluded = ll.load_returns(HIST_DIR, ASSETS)
print('excluded:', excluded)
print('rows', len(raw), '| assets', sorted(raw.underlying.unique()), '|', raw.state_time.min(), '->', raw.state_time.max())
panel = lm.dense_panel(raw)
del raw
panel, clock = lm.training_clock(panel)
labels, thresholds = lm.q99_labels(panel)
display(panel.groupby('split', sort=False).agg(days=('date','nunique'), rows=('v','size'), finite=('v','count'),
                                                first=('state_time','min'), last=('state_time','max')))
display(thresholds)
print(f'{time.time()-T0:.0f}s')

excluded: {'HYPE': '0 days'}
rows 1292538 | assets ['AVAX', 'BTC', 'ETH', 'SOL', 'TRX', 'XRP'] | 2024-09-01 00:05:00+00:00 -> 2026-09-18 23:55:00+00:00


,days,rows,finite,first,last
split,,,,,
train,448,774138,774132,2024-09-01 00:05:00+00:00,2025-11-22 23:55:00+00:00
valid,150,259200,259200,2025-11-23 00:00:00+00:00,2026-04-21 23:55:00+00:00
test,150,259200,259200,2026-04-22 00:00:00+00:00,2026-09-18 23:55:00+00:00


,underlying,train_q99_threshold_v
0,AVAX,0.000095
1,BTC,0.000026
2,ETH,0.000057
3,SOL,0.000081
4,TRX,0.000026
5,XRP,0.000124


8s


## 2. Features (K, A, B, E, F, C, U) on a common support

In [3]:
frame = panel.join(labels[['q99','onset','eligible_now'] + [f'y_{h}m' for h in lm.HORIZONS]])
frame = frame.join(lm.fixed_lags(panel).drop(columns='now').astype(np.float32))
frame['now'] = panel.y.astype(np.float32)
E = lm.simple_exponential_features(panel)
frame['ema_30m'] = E.ema_30m.astype(np.float32); frame['ema_240m'] = E.ema_240m.astype(np.float32)
config, Fconfig, Uconfig, cand = {}, {}, {}, []
for N in lm.N_GRID:
    rates, _ = lm.quadrature(.10, N)
    em = lm.ema_prior(panel, np.log(2)/rates)
    eq = lm.equal_exponential_features(panel, N, precomputed=em)
    e_key, s_key = f'f_n{N}_equal', f'f_n{N}_slope'
    frame[e_key] = eq.equal_memory.astype(np.float32); frame[s_key] = eq.fast_minus_slow.astype(np.float32)
    Fconfig[N] = ['now', e_key, s_key]; cand += [e_key, s_key]
    ucols = ll.add_unrestricted(frame, em, N)
    Uconfig[N] = ['now'] + ucols; cand += ucols
    for H in lm.H_GRID:
        lifted, info = lm.lifted_features(panel, H, N, precomputed=em)
        tag = f'h{int(100*H):02d}_n{N}'
        k, s = f'{tag}_kernel', f'{tag}_slope'
        frame[k] = lifted.kernel_memory.astype(np.float32); frame[s] = lifted.fast_minus_slow.astype(np.float32)
        cand += [k, s]
        config[tag] = dict(info, kernel_column=k, slope_column=s)
    del em
    print(f'N={N} done ({time.time()-T0:.0f}s)')
specs = {'K_climatology': [], 'A_current': ['now'],
         'B_fixed': ['now'] + [f'lag_{k*5}m' for k in lm.LAG_BARS],
         'E_exponential': ['now', 'ema_30m', 'ema_240m']}
core_cols = sorted(set(sum(specs.values(), []) + cand))
history = lm.memory_diagnostics(panel)
mask = history.memory_ready.to_numpy(bool) & np.isfinite(frame[core_cols].to_numpy(np.float32)).all(axis=1)
frame['eligible_now'] = frame.eligible_now.to_numpy(bool) & mask
display(panel[['split']].join(history).groupby('split').agg(history_coverage_median=('past_memory_coverage','median'),
        history_ready=('memory_ready','sum')))
display(frame.groupby('split').eligible_now.sum().rename('eligible_origins'))

N=4 done (40s)
N=6 done (56s)
N=8 done (71s)


,history_coverage_median,history_ready
split,,
test,1.0,259200
train,1.0,772410
valid,1.0,259200


split
test     258309
train    764674
valid    257456
Name: eligible_origins, dtype: int64

## 3. Validation-only selection (60 min log loss)

In [4]:
selection = []
def add(model, names, lam, **meta):
    m, _ = lm.fit_predict(frame, names, lam, lm.PRIMARY_HORIZON, eval_split='valid')
    selection.append(dict(model=model, ridge=lam, **meta, **m))
for model, names in specs.items():
    for lam in lm.RIDGE_GRID: add(model, names, lam, H=np.nan, N=np.nan, tag='')
for N, names in Fconfig.items():
    for lam in lm.RIDGE_GRID: add('F_equal_multiscale', names, lam, H=np.nan, N=N, tag=f'f_n{N}')
for N, names in Uconfig.items():
    for lam in lm.RIDGE_GRID: add('U_unrestricted', names, lam, H=np.nan, N=N, tag=f'u_n{N}')
for tag, meta in config.items():
    for lam in lm.RIDGE_GRID: add('C_lifted', ['now', meta['kernel_column'], meta['slope_column']], lam, H=meta['H'], N=meta['N'], tag=tag)
selection = pd.DataFrame(selection)
chosen = selection.sort_values(['logloss', 'ridge']).groupby('model', sort=False).head(1).set_index('model')
selection.to_csv(OUT/'12_validation_grid.csv', index=False, float_format='%.8g')
display(chosen[['ridge','H','N','tag','logloss','brier','positive_origin_count','n_origins']])
h_profile = (selection[selection.model == 'C_lifted'].groupby('H').logloss.min().rename('best_valid_logloss').reset_index())
h_profile['minus_best'] = h_profile.best_valid_logloss - h_profile.best_valid_logloss.min()
h_profile.to_csv(OUT/'12_h_profile.csv', index=False, float_format='%.8g')
print('Validation profile over H (descriptive: is H identified by forecasting?)'); display(h_profile)
chosen_tag = str(chosen.loc['C_lifted', 'tag']); chosen_info = config[chosen_tag]
chosen_specs = {**specs, 'F_equal_multiscale': Fconfig[int(chosen.loc['F_equal_multiscale', 'N'])],
                'C_lifted': ['now', chosen_info['kernel_column'], chosen_info['slope_column']],
                'U_unrestricted': Uconfig[int(chosen.loc['U_unrestricted', 'N'])]}
print(f'{time.time()-T0:.0f}s')

,ridge,H,N,tag,logloss,brier,positive_origin_count,n_origins
model,,,,,,,,
F_equal_multiscale,0.0100,NaN,4.0,f_n4,0.171374,0.044132,13343,257384
C_lifted,0.0100,0.1,6.0,h10_n6,0.171467,0.044112,13343,257384
E_exponential,0.0100,NaN,NaN,,0.172357,0.044140,13343,257384
U_unrestricted,0.0100,NaN,4.0,u_n4,0.173139,0.044099,13343,257384
B_fixed,0.0010,NaN,NaN,,0.178912,0.045316,13343,257384
A_current,0.0001,NaN,NaN,,0.189319,0.047167,13343,257384
K_climatology,0.0001,NaN,NaN,,0.195666,0.048168,13343,257384


Validation profile over H (descriptive: is H identified by forecasting?)


,H,best_valid_logloss,minus_best
0,0.05,0.171467,3.756781e-07
1,0.10,0.171467,0.000000e+00
2,0.20,0.171467,4.235164e-07
3,0.30,0.171470,3.708649e-06
4,0.40,0.171477,1.045828e-05


261s


## 4. Untouched test period, four horizons

In [5]:
rows, predictions = [], {}
for h in lm.HORIZONS:
    for model in ll.MODELS:
        lam = float(chosen.loc[model, 'ridge'])
        m, preds = lm.fit_predict(frame, chosen_specs[model], lam, h, eval_split='test')
        rows.append({'horizon_min': h, 'model': model, **m}); predictions[h, model] = preds
scores = pd.DataFrame(rows)
k = scores[scores.model == 'K_climatology'].set_index('horizon_min')
scores['logloss_skill_vs_K_pct'] = [100 * (1 - r.logloss / k.loc[r.horizon_min, 'logloss']) for r in scores.itertuples()]
scores['brier_skill_vs_K_pct'] = [100 * (1 - r.brier / k.loc[r.horizon_min, 'brier']) for r in scores.itertuples()]
scores.to_csv(OUT/'12_test_scores.csv', index=False, float_format='%.9g')
display(scores.pivot(index='horizon_min', columns='model', values='logloss').round(5))
print('Log-loss skill vs climatology K (%):')
display(scores.pivot(index='horizon_min', columns='model', values='logloss_skill_vs_K_pct').round(2))
print('Positive origin counts overlap (one onset labels many origins); they are not independent events.')

model,A_current,B_fixed,C_lifted,E_exponential,F_equal_multiscale,K_climatology,U_unrestricted
horizon_min,,,,,,,
30,0.08187,0.07317,0.07033,0.07038,0.07041,0.09035,0.07059
60,0.13360,0.11996,0.11447,0.11456,0.11434,0.14506,0.11508
90,0.17411,0.15671,0.14956,0.14975,0.14923,0.18797,0.15042
120,0.20847,0.18813,0.17940,0.17962,0.17889,0.22362,0.18044


Log-loss skill vs climatology K (%):


model,A_current,B_fixed,C_lifted,E_exponential,F_equal_multiscale,K_climatology,U_unrestricted
horizon_min,,,,,,,
30,9.38,19.01,22.16,22.10,22.06,0.0,21.87
60,7.90,17.31,21.09,21.03,21.18,0.0,20.67
90,7.37,16.63,20.43,20.33,20.61,0.0,19.98
120,6.77,15.87,19.77,19.68,20.00,0.0,19.31


Positive origin counts overlap (one onset labels many origins); they are not independent events.


## 5. Paired UTC-day bootstrap

In [6]:
comps = []
for h in lm.HORIZONS:
    for left, right in ll.COMPARISONS:
        pair = lm.align_comparison(predictions[h, left], predictions[h, right])
        comps.append({'horizon_min': h, 'comparison': f'{left} - {right}', 'n_paired_origins': len(pair),
                      **lm.bootstrap_day_diff(pair, B=BOOT, seed=1205 + h)})
boot = pd.DataFrame(comps)
boot.to_csv(OUT/'12_day_bootstrap.csv', index=False, float_format='%.9g')
display(boot[boot.horizon_min == lm.PRIMARY_HORIZON].round(6))

,horizon_min,comparison,n_paired_origins,delta_logloss,bootstrap_q025,bootstrap_q975,test_day_count
15,60,C_lifted - B_fixed,258237,-0.005490,-0.007700,-0.003500,150
16,60,C_lifted - E_exponential,258237,-0.000094,-0.000654,0.000531,150
17,60,C_lifted - F_equal_multiscale,258237,0.000130,-0.000211,0.000519,150
18,60,C_lifted - U_unrestricted,258237,-0.000611,-0.001331,0.000126,150
19,60,U_unrestricted - E_exponential,258237,0.000517,0.000162,0.000899,150
20,60,E_exponential - B_fixed,258237,-0.005396,-0.007892,-0.003162,150
21,60,F_equal_multiscale - B_fixed,258237,-0.005620,-0.007628,-0.003918,150
22,60,U_unrestricted - B_fixed,258237,-0.004879,-0.007617,-0.002395,150
23,60,B_fixed - A_current,258237,-0.013645,-0.017207,-0.010527,150
24,60,A_current - K_climatology,258237,-0.011460,-0.013526,-0.009554,150


## 6. Decision (60 min)

In [7]:
n_onsets = int(labels.loc[(labels.split == 'test') & labels.onset & mask, 'state_time'].nunique())
n_days = int(boot[boot.horizon_min == lm.PRIMARY_HORIZON].test_day_count.min())
decision = {**ll.decide(boot, n_onsets, n_days), 'selected_C': chosen_tag, 'selected_H': float(chosen_info['H']),
            'selected_N_C': int(chosen_info['N']), 'selected_N_U': int(chosen.loc['U_unrestricted', 'N']),
            'selected_N_F': int(chosen.loc['F_equal_multiscale', 'N'])}
(OUT/'12_decision.json').write_text(json.dumps(decision, indent=2) + '\n', encoding='utf-8')
print(json.dumps(decision, indent=2))

{
  "status": "TWO_EMA_MEMORY_SUFFICIENT",
  "primary_horizon_min": 60,
  "n_unique_test_onset_times": 445,
  "n_test_days": 150,
  "memory_vs_lags": true,
  "rough_specific": false,
  "beyond_two_ema": false,
  "two_ema_vs_lags": true,
  "skill": true,
  "vd11_rule_pass": false,
  "selected_C": "h10_n6",
  "selected_H": 0.1,
  "selected_N_C": 6,
  "selected_N_U": 4,
  "selected_N_F": 4
}


## 7. Manifest

In [8]:
manifest = {'module': 'Volatility_Dynamics_12', 'version': ll.VERSION, 'base_module_version': 'volatility_lifted_memory v0.1.1',
            'source': str(HIST_DIR), 'excluded_assets': excluded, 'assets': sorted(panel.underlying.unique()),
            'start': str(panel.state_time.min()), 'end': str(panel.state_time.max()),
            'split': {s: {'start': str(d.state_time.min()), 'end': str(d.state_time.max()), 'days': int(d.date.nunique())}
                      for s, d in panel.groupby('split')},
            'eligible_origins': frame.groupby('split').eligible_now.sum().astype(int).to_dict(),
            'train_q99_thresholds': dict(zip(thresholds.underlying, thresholds.train_q99_threshold_v)),
            'selection': {m: {'ridge': float(r.ridge), 'H': None if pd.isna(r.H) else float(r.H),
                              'N': None if pd.isna(r.N) else int(r.N), 'tag': str(r.tag)} for m, r in chosen.iterrows()},
            'chosen_kernel': chosen_info, 'decision': decision, 'registration': ll.registration(),
            'runtime_s': round(time.time() - T0)}
(OUT/'12_run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str) + '\n', encoding='utf-8')
print('written to', OUT, '| verdict:', decision['status'], f'| {time.time()-T0:.0f}s')

written to D:\Projects\deribit_market_data\notebooks\Volatility_Dynamics\12 | verdict: TWO_EMA_MEMORY_SUFFICIENT | 450s
